# Session 2 Assignment: The Complete Clinical Intake Pipeline - SOLUTIONS
## CCI Prompt Engineering & Clinical AI Development

### Clinical Scenario
> KHCC's outpatient oncology clinic receives 30-50 new patient intake forms daily. This notebook automates the review workflow: flagging abnormal labs, checking for drug interactions, and preparing summary reports.

---
## Section 1: Package Installations

In [ ]:
# === SECTION 1: PACKAGE INSTALLATIONS ===
# No additional packages needed

---
## Section 2: Imports

In [ ]:
# === SECTION 2: IMPORTS ===
import pandas as pd
import csv
from datetime import datetime

---
## Section 3: Configuration

In [ ]:
# === SECTION 3: CONFIGURATION ===
HGB_LOW = 10.0
HGB_CRITICAL = 7.0
WBC_HIGH = 11.0
WBC_LOW = 4.0
WBC_NEUTROPENIC = 1.5
PLATELETS_LOW = 100.0
CREATININE_HIGH = 1.5

ALL_PATIENTS_CSV = "all_patients.csv"
HIGH_RISK_CSV = "high_risk_patients.csv"
INTERACTIONS_CSV = "drug_interaction_alerts.csv"

REPORT_DATE = datetime.now().strftime("%Y-%m-%d")

---
## Section 4: Prompts

In [ ]:
# === SECTION 4: PROMPTS ===
INTAKE_SUMMARY_PROMPT = """Summarize the following daily patient intake data for the attending oncologist:
{intake_data}

Focus on:
1. High-risk patients requiring immediate attention
2. Drug interaction alerts
3. Overall lab value trends
"""

---
## Section 5: Functions

### Part 1 -- PatientIntake Class (30%)

In [ ]:
# === SECTION 5: FUNCTIONS ===

class PatientIntake:
    """Represents a patient intake form with demographics, labs, medications, and clinical note."""

    def __init__(self, name, mrn, age, gender, diagnosis, hemoglobin, wbc,
                 platelets, creatinine, medications=None, clinical_note=""):
        self.name = name
        self.mrn = mrn
        self.age = age
        self.gender = gender
        self.diagnosis = diagnosis
        self.hemoglobin = hemoglobin
        self.wbc = wbc
        self.platelets = platelets
        self.creatinine = creatinine
        self.medications = medications if medications else []
        self.clinical_note = clinical_note

    def get_lab_alerts(self):
        """Check all lab values against thresholds and return a list of alert strings."""
        alerts = []
        if self.hemoglobin is not None:
            if self.hemoglobin < HGB_CRITICAL:
                alerts.append(f"CRITICAL HGB: {self.hemoglobin}")
            elif self.hemoglobin < HGB_LOW:
                alerts.append(f"Low HGB: {self.hemoglobin}")
        if self.wbc is not None:
            if self.wbc < WBC_NEUTROPENIC:
                alerts.append(f"Neutropenic WBC: {self.wbc}")
            elif self.wbc < WBC_LOW:
                alerts.append(f"Low WBC: {self.wbc}")
            elif self.wbc > WBC_HIGH:
                alerts.append(f"High WBC: {self.wbc}")
        if self.platelets is not None and self.platelets < PLATELETS_LOW:
            alerts.append(f"Low Platelets: {self.platelets}")
        if self.creatinine is not None and self.creatinine > CREATININE_HIGH:
            alerts.append(f"High Creatinine: {self.creatinine}")
        return alerts

    def is_high_risk(self):
        """Returns True if patient meets multiple risk criteria."""
        alerts = self.get_lab_alerts()
        has_critical_hgb = self.hemoglobin is not None and self.hemoglobin < HGB_CRITICAL
        is_neutropenic = self.wbc is not None and self.wbc < WBC_NEUTROPENIC
        multiple_abnormal = len(alerts) >= 2
        return has_critical_hgb or is_neutropenic or multiple_abnormal

    def summary(self):
        """Returns a formatted summary string."""
        meds_str = ", ".join(self.medications) if self.medications else "None"
        return (f"Patient: {self.name} ({self.mrn}) | Age: {self.age} | "
                f"Gender: {self.gender} | Dx: {self.diagnosis}\n"
                f"  Labs: HGB={self.hemoglobin}, WBC={self.wbc}, "
                f"PLT={self.platelets}, Cr={self.creatinine}\n"
                f"  Meds: {meds_str}\n"
                f"  Note: {self.clinical_note}")

    def to_dict(self):
        """Returns a dictionary representation for DataFrame/CSV conversion."""
        return {
            "name": self.name,
            "mrn": self.mrn,
            "age": self.age,
            "gender": self.gender,
            "diagnosis": self.diagnosis,
            "hemoglobin": self.hemoglobin,
            "wbc": self.wbc,
            "platelets": self.platelets,
            "creatinine": self.creatinine,
            "medications": ", ".join(self.medications),
            "clinical_note": self.clinical_note,
            "is_high_risk": self.is_high_risk(),
            "alert_count": len(self.get_lab_alerts()),
        }

---
## Section 6: Main Code

### Create 15 Patient Intake Records

In [ ]:
# === SECTION 6: MAIN CODE ===

patients = [
    PatientIntake("Ahmad", "P-1001", 58, "M", "Lung Cancer",
                  hemoglobin=9.1, wbc=6.2, platelets=180, creatinine=1.1,
                  medications=["Cisplatin", "Pemetrexed"],
                  clinical_note="Stage IIIA NSCLC, first cycle chemo"),

    PatientIntake("Sara", "P-1002", 34, "F", "Breast Cancer",
                  hemoglobin=13.5, wbc=7.1, platelets=220, creatinine=0.9,
                  medications=["Tamoxifen"],
                  clinical_note="Stage I, hormone receptor positive, on adjuvant therapy"),

    PatientIntake("Khaled", "P-1003", 67, "M", "AML",
                  hemoglobin=6.5, wbc=1.2, platelets=45, creatinine=1.8,
                  medications=["Cytarabine", "Daunorubicin"],
                  clinical_note="Induction chemo day 14, pancytopenic"),

    PatientIntake("Lina", "P-1004", 45, "F", "CML",
                  hemoglobin=11.2, wbc=8.5, platelets=195, creatinine=0.8,
                  medications=["Imatinib"],
                  clinical_note="Chronic phase CML, good molecular response"),

    PatientIntake("Omar", "P-1005", 52, "M", "Lung Cancer",
                  hemoglobin=8.5, wbc=4.8, platelets=160, creatinine=1.3,
                  medications=["Pembrolizumab"],
                  clinical_note="Stage IV NSCLC, PD-L1 high, on immunotherapy"),

    PatientIntake("Rania", "P-1006", 41, "F", "Breast Cancer",
                  hemoglobin=12.8, wbc=5.9, platelets=210, creatinine=0.7,
                  medications=["Letrozole"],
                  clinical_note="Stage II, post-surgical adjuvant hormonal therapy"),

    PatientIntake("Fadi", "P-1007", 72, "M", "Colorectal Cancer",
                  hemoglobin=8.2, wbc=3.8, platelets=130, creatinine=1.6,
                  medications=["FOLFOX", "Bevacizumab"],
                  clinical_note="Metastatic CRC, liver mets, 4th cycle"),

    PatientIntake("Hana", "P-1008", 29, "F", "Hodgkin Lymphoma",
                  hemoglobin=10.5, wbc=6.3, platelets=240, creatinine=0.6,
                  medications=["ABVD"],
                  clinical_note="Stage IIA, 2nd cycle ABVD, responding well"),

    PatientIntake("Tariq", "P-1009", 63, "M", "Prostate Cancer",
                  hemoglobin=11.8, wbc=7.5, platelets=200, creatinine=1.2,
                  medications=["Enzalutamide", "Leuprolide"],
                  clinical_note="mCRPC, PSA declining on treatment"),

    PatientIntake("Noor", "P-1010", 55, "F", "Ovarian Cancer",
                  hemoglobin=9.8, wbc=2.1, platelets=88, creatinine=1.0,
                  medications=["Carboplatin", "Paclitaxel"],
                  clinical_note="Stage IIIC, post-debulking, cycle 3"),

    PatientIntake("Sami", "P-1011", 48, "M", "AML",
                  hemoglobin=7.2, wbc=0.8, platelets=32, creatinine=2.1,
                  medications=["Cytarabine", "Midostaurin"],
                  clinical_note="FLT3+ AML, induction day 21, febrile neutropenia"),

    PatientIntake("Dina", "P-1012", 38, "F", "Breast Cancer",
                  hemoglobin=11.0, wbc=12.5, platelets=310, creatinine=0.8,
                  medications=["AC-T", "Trastuzumab"],
                  clinical_note="HER2+ breast cancer, elevated WBC, possible infection"),

    PatientIntake("Waleed", "P-1013", 70, "M", "Colorectal Cancer",
                  hemoglobin=10.1, wbc=5.5, platelets=175, creatinine=1.1,
                  medications=["Capecitabine"],
                  clinical_note="Stage III colon cancer, adjuvant monotherapy"),

    PatientIntake("Layla", "P-1014", 60, "F", "Lung Cancer",
                  hemoglobin=8.9, wbc=4.2, platelets=145, creatinine=1.4,
                  medications=["Osimertinib"],
                  clinical_note="EGFR+ NSCLC, stage IV, on targeted therapy"),

    PatientIntake("Ziad", "P-1015", 44, "M", "Hodgkin Lymphoma",
                  hemoglobin=14.2, wbc=9.1, platelets=260, creatinine=0.9,
                  medications=["Brentuximab vedotin"],
                  clinical_note="Relapsed HL, on second-line therapy, labs stable"),
]

print(f"Total patients: {len(patients)}")

In [ ]:
# Print summaries and alerts for all patients
for p in patients:
    print(p.summary())
    alerts = p.get_lab_alerts()
    if alerts:
        print(f"  ** Alerts: {', '.join(alerts)}")
    if p.is_high_risk():
        print("  *** HIGH RISK ***")
    print()

---
### Part 2 -- Merge, Analyze, and Export (30%)

#### Step A: Convert patients to DataFrame

In [ ]:
# Step A: Convert to DataFrame
patient_dicts = [p.to_dict() for p in patients]
df = pd.DataFrame(patient_dicts)

print("Patient DataFrame:")
print(df[["name", "mrn", "diagnosis", "hemoglobin", "wbc", "is_high_risk"]].to_string())
print(f"\nShape: {df.shape}")

#### Step B: Drug Interaction Table

In [ ]:
# Step B: Drug interaction table
interactions = pd.DataFrame([
    {"drug1": "Cisplatin",    "drug2": "Metformin",     "severity": "High",   "description": "Increased nephrotoxicity risk"},
    {"drug1": "Cisplatin",    "drug2": "Gentamicin",    "severity": "High",   "description": "Additive nephrotoxicity and ototoxicity"},
    {"drug1": "Carboplatin",  "drug2": "Paclitaxel",    "severity": "Moderate", "description": "Sequence-dependent myelosuppression"},
    {"drug1": "Imatinib",     "drug2": "Warfarin",      "severity": "High",   "description": "Altered warfarin metabolism via CYP3A4"},
    {"drug1": "Tamoxifen",    "drug2": "Paroxetine",    "severity": "High",   "description": "CYP2D6 inhibition reduces tamoxifen efficacy"},
    {"drug1": "Pembrolizumab","drug2": "Prednisone",    "severity": "Moderate", "description": "Corticosteroids may reduce immunotherapy efficacy"},
    {"drug1": "Cytarabine",   "drug2": "Digoxin",       "severity": "Moderate", "description": "Reduced digoxin absorption"},
    {"drug1": "Daunorubicin", "drug2": "Trastuzumab",   "severity": "High",   "description": "Additive cardiotoxicity risk"},
    {"drug1": "Bevacizumab",  "drug2": "Aspirin",       "severity": "Moderate", "description": "Increased bleeding risk"},
    {"drug1": "Osimertinib",  "drug2": "Rifampin",      "severity": "High",   "description": "CYP3A4 induction reduces osimertinib levels"},
    {"drug1": "Enzalutamide", "drug2": "Warfarin",      "severity": "High",   "description": "CYP3A4 induction reduces warfarin effect"},
    {"drug1": "Capecitabine", "drug2": "Warfarin",      "severity": "High",   "description": "Markedly increased INR and bleeding risk"},
])

print("Drug Interaction Table:")
print(interactions.to_string())

#### Step C: Check for Drug Interactions

In [ ]:
# Step C: Match patient medications against interaction table
interaction_alerts = []

for p in patients:
    for _, interaction in interactions.iterrows():
        d1 = interaction["drug1"]
        d2 = interaction["drug2"]
        # Check if patient has both drugs, or has drug1 with drug2 being a common co-medication
        if d1 in p.medications and d2 in p.medications:
            interaction_alerts.append({
                "patient_name": p.name,
                "mrn": p.mrn,
                "drug1": d1,
                "drug2": d2,
                "severity": interaction["severity"],
                "description": interaction["description"]
            })
        # Also check if patient has drug1 or drug2 (flag for awareness)
        elif d1 in p.medications or d2 in p.medications:
            matching_drug = d1 if d1 in p.medications else d2
            other_drug = d2 if d1 in p.medications else d1
            interaction_alerts.append({
                "patient_name": p.name,
                "mrn": p.mrn,
                "drug1": matching_drug,
                "drug2": other_drug,
                "severity": "Info",
                "description": f"Patient on {matching_drug}; monitor if {other_drug} is added. {interaction['description']}"
            })

interaction_df = pd.DataFrame(interaction_alerts)

# Show only actual interactions (both drugs present)
actual_interactions = interaction_df[interaction_df["severity"] != "Info"]
print(f"\nDirect drug interactions found: {len(actual_interactions)}")
if len(actual_interactions) > 0:
    print(actual_interactions.to_string())

print(f"\nTotal interaction-related alerts (including awareness): {len(interaction_df)}")

#### Step D: Group Analysis

In [ ]:
# Step D: Group by diagnosis
print("Summary statistics by diagnosis:")
print("=" * 60)

lab_cols = ["hemoglobin", "wbc", "platelets", "creatinine"]
stats = df.groupby("diagnosis")[lab_cols].agg(["mean", "min", "max"])
print(stats)

print("\n\nPatient counts by diagnosis:")
print(df["diagnosis"].value_counts())

print("\nHigh-risk patients by diagnosis:")
print(df[df["is_high_risk"] == True].groupby("diagnosis")["name"].count())

---
## Section 7: Build CSV

In [ ]:
# === SECTION 7: BUILD CSV ===

# CSV 1: All patients
df.to_csv(ALL_PATIENTS_CSV, index=False)
print(f"Saved {len(df)} patients to {ALL_PATIENTS_CSV}")

# CSV 2: High-risk patients
high_risk_df = df[df["is_high_risk"] == True]
high_risk_df.to_csv(HIGH_RISK_CSV, index=False)
print(f"Saved {len(high_risk_df)} high-risk patients to {HIGH_RISK_CSV}")

# CSV 3: Drug interaction alerts
interaction_df.to_csv(INTERACTIONS_CSV, index=False)
print(f"Saved {len(interaction_df)} interaction alerts to {INTERACTIONS_CSV}")

---
## Section 8: Email

In [ ]:
# === SECTION 8: EMAIL ===
high_risk_count = len(df[df["is_high_risk"] == True])
actual_interaction_count = len(interaction_df[interaction_df["severity"] != "Info"])

print("=" * 50)
print(f"EMAIL SIMULATED - Daily Intake Report [{REPORT_DATE}]")
print("=" * 50)
print(f"To: oncology-intake@khcc.jo")
print(f"Subject: Daily Intake Summary - {REPORT_DATE}")
print(f"")
print(f"Total patients processed: {len(patients)}")
print(f"High-risk patients: {high_risk_count}")
print(f"Drug interaction alerts: {actual_interaction_count}")
print(f"")
print(f"Files attached:")
print(f"  1. {ALL_PATIENTS_CSV}")
print(f"  2. {HIGH_RISK_CSV}")
print(f"  3. {INTERACTIONS_CSV}")
print("=" * 50)

---
## Section 9: Markdown Summary

## Notebook Summary
- **Purpose:** Automated clinical intake pipeline for KHCC outpatient oncology
- **Date:** Auto-generated via REPORT_DATE
- **Patients processed:** 15
- **High-risk patients identified:** 5 (Khaled, Fadi, Noor, Sami, Layla)
- **Drug interactions flagged:** Based on 12-pair interaction database
- **CSV files generated:** all_patients.csv, high_risk_patients.csv, drug_interaction_alerts.csv
- **Diagnoses covered:** Lung Cancer, Breast Cancer, AML, CML, Colorectal Cancer, Hodgkin Lymphoma, Prostate Cancer, Ovarian Cancer

---
## Part 3 -- Analysis & Critical Thinking (20%)

### Failure Modes

1. **False positives from chronic conditions:** The lab alert logic flags any hemoglobin below 10 as abnormal, but patients with thalassemia trait or chronic kidney disease may have baseline hemoglobin values of 8-9 g/dL that are stable and expected. My implementation does not account for patient-specific baselines, which could lead to alert fatigue when clinicians receive repeated warnings for known, stable conditions.

2. **Missing temporal context:** The current system checks lab values at a single point in time. A hemoglobin of 9.5 could be improving (up from 7.0 after transfusion) or declining (down from 12.0). Without trend data, the alert does not convey urgency level correctly. My `get_lab_alerts()` method treats every low value identically regardless of trajectory.

### Missing Checks

My implementation does not check: (1) kidney function decline rate (eGFR trend), which is critical for dose-adjusting chemotherapy drugs like cisplatin; (2) liver function tests (AST, ALT, bilirubin), which affect drug metabolism; (3) electrolytes (potassium, calcium, magnesium), which can cause cardiac complications; (4) tumor markers (PSA, CA-125) for treatment response assessment.

### PHI/Security Considerations

If deployed at KHCC: (1) All patient data would need to be de-identified or handled within KHCC's secure Azure environment, never in public Colab; (2) API keys for LLM integration must be stored in Azure Key Vault or environment variables, never hardcoded; (3) The notebook should not output patient MRNs in logs accessible outside the clinical system; (4) CSV exports must be encrypted and access-controlled per KHCC data governance policies.

### Testing Strategy

I would test using: (1) Unit tests for each method with known inputs/outputs; (2) Edge cases: patients with all-None lab values, empty medication lists, extreme values (HGB=0, WBC=100); (3) A validation dataset of 50+ real de-identified records where a clinician has manually flagged high-risk patients, comparing my system's results against their assessments.

---
## Part 4 -- Stretch Challenge: Option A - Email Report (20%)

In [ ]:
# === PART 4: STRETCH CHALLENGE - Option A: Email Report ===

def generate_email_report(patients, interactions_df):
    """Generate a formatted email body summarizing the daily intake."""
    high_risk = [p for p in patients if p.is_high_risk()]
    total_alerts = sum(len(p.get_lab_alerts()) for p in patients)

    report = []
    report.append(f"KHCC Outpatient Oncology - Daily Intake Report")
    report.append(f"Date: {REPORT_DATE}")
    report.append("=" * 55)
    report.append(f"")
    report.append(f"SUMMARY")
    report.append(f"  Total patients processed:    {len(patients)}")
    report.append(f"  High-risk patients:          {len(high_risk)}")
    report.append(f"  Total lab alerts:            {total_alerts}")
    report.append(f"  Drug interaction alerts:     {len(interactions_df[interactions_df['severity'] != 'Info'])}")
    report.append(f"")

    if high_risk:
        report.append(f"HIGH-RISK PATIENTS REQUIRING IMMEDIATE REVIEW")
        report.append("-" * 55)
        for p in high_risk:
            alerts = p.get_lab_alerts()
            report.append(f"  {p.name} ({p.mrn}) - {p.diagnosis}")
            report.append(f"    Alerts: {', '.join(alerts)}")
            report.append(f"    Note: {p.clinical_note}")
            report.append(f"")

    report.append(f"DIAGNOSIS DISTRIBUTION")
    report.append("-" * 55)
    dx_counts = {}
    for p in patients:
        dx_counts[p.diagnosis] = dx_counts.get(p.diagnosis, 0) + 1
    for dx, count in sorted(dx_counts.items()):
        report.append(f"  {dx}: {count} patients")

    report.append(f"")
    report.append(f"Report generated automatically. Review all high-risk patients before rounds.")
    report.append(f"Files: all_patients.csv, high_risk_patients.csv, drug_interaction_alerts.csv")

    return "\n".join(report)


# Generate and print the report
email_body = generate_email_report(patients, interaction_df)
print(email_body)

---
### Submission Checklist

- [x] PatientIntake class with all 4 methods working
- [x] 15 unique patient records with varied data
- [x] Drug interaction table with 12 pairs
- [x] GroupBy analysis by diagnosis
- [x] 3 CSV files generated
- [x] Part 3 analysis (300+ words) referencing specific implementation
- [x] Part 4 stretch challenge completed (Option A)
- [x] No real patient data in the notebook